# Instagram Engagement Prediction and Analysis Notebook
This notebook demonstrates a complete workflow for predicting and analyzing sentiment-weighted engagement on Instagram posts using machine learning and deep learning models. It includes data preprocessing, feature engineering, model training (XGBoost, LightGBM, DNN), evaluation, visualizations, SHAP explainability, and a keyword recommendation system.

In [ ]:
# Install required packages for Colab
def install_packages():
    import sys
    !{sys.executable} -m pip install pandas numpy scikit-learn xgboost lightgbm tensorflow transformers matplotlib seaborn shap nltk wordcloud
    import nltk
    nltk.download('punkt')

install_packages()

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
import tensorflow as tf
from transformers import BertTokenizer, TFBertModel
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
from google.colab import files
nltk.download('punkt')

## Upload your Instagram dataset
Please upload your Instagram dataset CSV file. The file should contain columns such as captions, hashtags, engagement metrics, and sentiment scores.

In [ ]:
# Upload dataset
print("Please upload your Instagram dataset CSV file:")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
data = pd.read_csv(file_name)

In [ ]:
# Preprocessing: Handle missing values
data['caption'] = data['caption'].fillna('')
data['hashtags'] = data['hashtags'].fillna('')
data['comment_text'] = data['comment_text'].fillna('')
data = data.fillna(0)

# Convert timestamp to datetime
data['timestamp'] = pd.to_datetime(data['timestamp'], unit='s')

In [ ]:
# Define features to exclude (sentiment-related and target)
exclude_cols = [
    'sentiment', 'sentiment_confidence', 'sentiment_positive', 'sentiment_negative', 
    'sentiment_neutral', 'sentiment_positive_post', 'sentiment_negative_post', 
    'sentiment_neutral_post', 'avg_comment_sentiment', 'sentiment_weighted_engagement'
]

# Feature selection
features = [
    'likes', 'comments_count', 'caption', 'hashtags', 'media_type', 'is_private', 
    'is_verified', 'caption_length', 'num_hashtags', 'has_mention', 'has_url', 
    'follower_adjusted_likes', 'follower_adjusted_comments', 'engagement_rate', 
    'engagement_frequency', 'influence_score', 'content_interaction', 
    'comment_engagement_ratio', 'comment_length', 'has_emoji', 
    'category_beauty', 'category_family', 'category_fashion', 'category_fitness', 
    'category_food', 'category_pet', 'category_travel', '#Followers', 
    '#Followees', '#Posts', 'num_comments', 'comment_likes_sum'
]

X = data[features]
y = data['sentiment_weighted_engagement']

In [ ]:
# Encode categorical variables
le_media_type = LabelEncoder()
X['media_type'] = le_media_type.fit_transform(X['media_type'])

In [ ]:
# Text Processing: TF-IDF for captions and hashtags
tfidf_caption = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_hashtags = TfidfVectorizer(max_features=50, token_pattern=r'#\\w+')

caption_tfidf = tfidf_caption.fit_transform(X['caption']).toarray()
hashtags_tfidf = tfidf_hashtags.fit_transform(X['hashtags']).toarray()

In [ ]:
# BERT Embeddings for Captions
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = TFBertModel.from_pretrained('bert-base-uncased')

def get_bert_embeddings(texts, max_length=128):
    inputs = tokenizer(texts.tolist(), return_tensors='tf', max_length=max_length, 
                      padding=True, truncation=True)
    outputs = bert_model(inputs)
    return outputs.pooler_output.numpy()

bert_embeddings = get_bert_embeddings(X['caption'])

In [ ]:
# Numerical and Categorical Features
numerical_cols = [
    'likes', 'comments_count', 'caption_length', 'num_hashtags', 
    'follower_adjusted_likes', 'follower_adjusted_comments', 'engagement_rate', 
    'engagement_frequency', 'influence_score', 'content_interaction', 
    'comment_engagement_ratio', 'comment_length', '#Followers', '#Followees', 
    '#Posts', 'num_comments', 'comment_likes_sum'
]
categorical_cols = [
    'media_type', 'is_private', 'is_verified', 'has_mention', 'has_url', 
    'category_beauty', 'category_family', 'category_fashion', 'category_fitness', 
    'category_food', 'category_pet', 'category_travel'
]

# Scale numerical features
scaler = StandardScaler()
X_numerical = scaler.fit_transform(X[numerical_cols])

In [ ]:
# Combine all features
X_combined = np.hstack([
    X_numerical,
    X[categorical_cols].values,
    caption_tfidf,
    hashtags_tfidf,
    bert_embeddings
])

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

In [ ]:
# Model 1: XGBoost
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=6)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [ ]:
# Model 2: LightGBM
lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=6)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)

In [ ]:
# Model 3: Deep Neural Network
input_shape = X_combined.shape[1]
dnn_model = Sequential([
    Input(shape=(input_shape,)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])
dnn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
dnn_model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
dnn_pred = dnn_model.predict(X_test).flatten()

In [ ]:
# Evaluate Models
def evaluate_model(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} Performance:")
    print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")
    return mse, mae, r2

xgb_metrics = evaluate_model(y_test, xgb_pred, "XGBoost")
lgb_metrics = evaluate_model(y_test, lgb_pred, "LightGBM")
dnn_metrics = evaluate_model(y_test, dnn_pred, "DNN")

## Visualizations
The following cells provide visualizations for data exploration, model evaluation, and feature importance.

In [ ]:
# 1. Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data[numerical_cols + ['sentiment_weighted_engagement']].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# 2. Engagement Distribution
plt.figure(figsize=(10, 6))
sns.histplot(y, bins=30, kde=True)
plt.title("Distribution of Sentiment-Weighted Engagement")
plt.xlabel("Sentiment-Weighted Engagement")
plt.show()

In [ ]:
# 3. Feature Importance Plot for XGBoost
feature_importance = xgb_model.feature_importances_[:len(numerical_cols)]

In [ ]:
# 4. Actual vs Predicted Scatter Plots
# XGBoost
plt.figure(figsize=(8, 6))
plt.scatter(y_test, xgb_pred, alpha=0.5, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Engagement")
plt.ylabel("Predicted Engagement")
plt.title("XGBoost: Actual vs Predicted")
plt.show()

# LightGBM
plt.figure(figsize=(8, 6))
plt.scatter(y_test, lgb_pred, alpha=0.5, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Engagement")
plt.ylabel("Predicted Engagement")
plt.title("LightGBM: Actual vs Predicted")
plt.show()

# DNN
plt.figure(figsize=(8, 6))
plt.scatter(y_test, dnn_pred, alpha=0.5, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Engagement")
plt.ylabel("Predicted Engagement")
plt.title("DNN: Actual vs Predicted")
plt.show()

In [ ]:
# 5. Box Plots for Numerical Features
plt.figure(figsize=(15, 8))
X[numerical_cols].boxplot()
plt.xticks(rotation=45)
plt.title("Box Plots of Numerical Features")
plt.show()

In [ ]:
# 6. Bar Plot of Engagement by Categorical Features
# Media Type
media_engagement = data.groupby('media_type')['sentiment_weighted_engagement'].mean().reset_index()
media_labels = le_media_type.inverse_transform(media_engagement['media_type'])

In [ ]:
# Categories
category_cols = ['category_beauty', 'category_family', 'category_fashion', 'category_fitness', 
                 'category_food', 'category_pet', 'category_travel']
category_means = [data[data[col] == 1]['sentiment_weighted_engagement'].mean() for col in category_cols]
category_means = [0 if pd.isna(x) else x for x in category_means]  # Handle NaN

In [ ]:
# 7. Time Series Plot
time_data = data.groupby(data['timestamp'].dt.date)['sentiment_weighted_engagement'].mean().reset_index()

In [ ]:
# 8. Word Cloud for Captions and Hashtags
caption_text = ' '.join(data['caption'].str.lower())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(caption_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud of Captions")
plt.show()

hashtag_text = ' '.join(data['hashtags'].str.lower())
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(hashtag_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud of Hashtags")
plt.show()

In [ ]:
# 9. Residual Plots
# XGBoost
plt.figure(figsize=(8, 6))
plt.scatter(xgb_pred, y_test - xgb_pred, alpha=0.5, color='blue')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Engagement")
plt.ylabel("Residuals")
plt.title("XGBoost: Residual Plot")
plt.show()

# LightGBM
plt.figure(figsize=(8, 6))
plt.scatter(lgb_pred, y_test - lgb_pred, alpha=0.5, color='green')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Engagement")
plt.ylabel("Residuals")
plt.title("LightGBM: Residual Plot")
plt.show()

# DNN
plt.figure(figsize=(8, 6))
plt.scatter(dnn_pred, y_test - dnn_pred, alpha=0.5, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Engagement")
plt.ylabel("Residuals")
plt.title("DNN: Residual Plot")
plt.show()

In [ ]:
# 10. SHAP Summary Plot
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=(
    numerical_cols + categorical_cols + 
    [f"caption_tfidf_{i}" for i in range(caption_tfidf.shape[1])] + 
    [f"hashtag_tfidf_{i}" for i in range(hashtags_tfidf.shape[1])] + 
    [f"bert_{i}" for i in range(bert_embeddings.shape[1])]
))
plt.title("SHAP Summary Plot for XGBoost")
plt.tight_layout()
plt.show()

In [ ]:
# Keyword Recommendation System
def recommend_keywords(caption, hashtags, top_n=5):
    all_captions = data['caption'].str.lower().str.cat(sep=' ')
    all_hashtags = data['hashtags'].str.lower().str.cat(sep=' ')
    caption_tokens = word_tokenize(caption.lower())
    hashtag_tokens = word_tokenize(hashtags.lower())
    
    caption_words = Counter(word_tokenize(all_captions))
    hashtag_words = Counter(word_tokenize(all_hashtags))
    
    caption_recommend = [word for word, _ in caption_words.most_common(top_n) if word not in caption_tokens]
    hashtag_recommend = [word for word, _ in hashtag_words.most_common(top_n) if word not in hashtag_tokens]
    
    return caption_recommend, hashtag_recommend

# Example recommendation
sample_caption = "In a World where folks rarely want to see you change, KEEP GETTIN' MONEY!"
sample_hashtags = "#bagsecured"
caption_rec, hashtag_rec = recommend_keywords(sample_caption, sample_hashtags)
print("Recommended Keywords for Caption:", caption_rec)
print("Recommended Hashtags:", hashtag_rec)

In [ ]:
# Predict Engagement for New Input
def predict_engagement(caption, hashtags, numerical_features, categorical_features):
    caption_tfidf_new = tfidf_caption.transform([caption]).toarray()
    hashtags_tfidf_new = tfidf_hashtags.transform([hashtags]).toarray()
    bert_new = get_bert_embeddings(pd.Series([caption]))
    
    numerical_scaled = scaler.transform([numerical_features])
    
    input_features = np.hstack([
        numerical_scaled,
        [categorical_features],
        caption_tfidf_new,
        hashtags_tfidf_new,
        bert_new
    ])
    
    pred = xgb_model.predict(input_features)[0]
    return pred

# Example prediction
sample_numerical = X[numerical_cols].iloc[0].values
sample_categorical = X[categorical_cols].iloc[0].values
pred_engagement = predict_engagement(sample_caption, sample_hashtags, sample_numerical, sample_categorical)
print(f"Predicted Sentiment-Weighted Engagement: {pred_engagement:.4f}")

## Recommendations for High Engagement
- Use engaging captions with positive tone and clear calls-to-action.
- Include popular hashtags relevant to your category.
- Post during peak engagement times based on timestamp analysis.
- Leverage high-value followers by encouraging their interaction.